## Autoformatting_tool



### The blueprint: Class tool
~a blueprint for every tool.
* It says: Every tool box must have a name, a description, a func (the callable), arguments, and outputs

* This generic Tool class can be reused whenever we need to use an actual tool includes:
1. name (str): The name of the tool.
2. description (str): A brief description of what the tool does.
3. function (callable): anything you can call with parentheses like a function" (e.g., my_function()).The function the tool executes.
4. arguments (list): The expected input parameters.
5. outputs (str or list): The expected outputs of the tool.
6. __call__(): Calls the function when the tool instance is invoked.
7. to_string(): Converts the tool’s attributes into a textual representation.

In [214]:
from typing import Callable

class Tool:
    def __init__(self,
        name: str,
        description: str,
        func: Callable,
        arguments: list,
        outputs: str):
        self.name = name
        self.description = description
        self.func = func
        self.arguments = arguments
        self.outputs = outputs
        
    def to_string(self) -> str:
        """Return a string representation of the tool, including its name, description, arguments, and outputs."""
        args_str = ", ".join([f"{arg_name}: {arg_type}" for arg_name, arg_type in self.arguments])
        return f"Tool Name: {self.name}, Description: {self.description}, Arguments: {args_str}, Outputs: {self.outputs}"

    def __call__(self, *args, **kwargs):
        return self.func(*args, **kwargs)

## The Decorator
def tool(func)
- It takes your calculator function.
- It inspects the code (inspect module) to read the name (calculator), the docstring ("Multiply two integers."), and the inputs (a, b).
- It automatically builds a brand-new Tool object using the Tool class blueprint.
@ tool is a decorator indicating a function to  be passed into the tool class then the tool function to be inspected

In [218]:
import inspect

def tool(func):
    signature = inspect.signature(func)

    # Extract argument names and types
    arguments = []
    for param in signature.parameters.values():
        annotation_name = (
            param.annotation.__name__
            if hasattr(param.annotation, '__name__')
            else str(param.annotation)
        )
        arguments.append((param.name, annotation_name))

    return_annotation = signature.return_annotation
    if return_annotation is inspect._empty:
        outputs = "No return annotation"
    else:
        outputs = (
            return_annotation.__name__
            if hasattr(return_annotation, '__name__')
            else str(return_annotation)
        )

    description = func.__doc__ or "No description provided."
    name = func.__name__

    # It wraps your function into a Tool class instance!
    return Tool(
        name=name,
        description=description,
        func=func,
        arguments=arguments,
        outputs=outputs
    )

* The Raw Function (calculator):
This is just standard Python code that does a job. It doesn't know anything about LLMs; it just multiplies numbers.

In [222]:
@tool
def calculator(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

print(calculator.to_string())

Tool Name: calculator, Description: Multiply two integers., Arguments: a: int, b: int, Outputs: int


Our tool was written in Python, and the implementation already provides everything we need:

1. A descriptive name of what it does: calculator
2. A longer description, provided by the function’s docstring comment: Multiply two integers.
3. The inputs and their type: the function clearly expects two ints.
4. The type of the output.

Note:
#### Type Hints / The typing Module (Static analysis)

When Python developers talk about "typing" or the typing module, they are usually referring to type hints. This is an optional feature that allows you to explicitly state what data types you expect your variables, function arguments, and return values to be

